In [1]:
import random
import numpy as np
from osgeo import gdal
from osgeo import gdalconst
from osgeo.gdalconst import *
try:
            from osgeo import ogr
except:
            import ogr
from sklearn.model_selection import RepeatedKFold
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split,cross_val_score,cross_validate,GridSearchCV
from sklearn import preprocessing
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,recall_score,f1_score,classification_report,roc_curve,auc

import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier

import pandas as pd
import cv2

In [5]:
path = r'E:\eaasy\Thesis\水体边界提取文章_apply science\副本\修改资料\Data\Data2019_10_20\TrueClassificationResult\TrueResultArea2\Area2TrueResultFromMajority5X5.dat'
#gdal.Register()
dataset = gdal.Open(path)    

geoTransform = dataset.GetGeoTransform()

proj = dataset.GetProjection()

cols = dataset.RasterXSize

rows = dataset.RasterYSize

band = dataset.GetRasterBand(1)


y_true = band.ReadAsArray(0,0,cols,rows)


In [6]:
y_true = np.array(y_true,dtype=np.int16) 

In [7]:
for i in range(0,y_true.shape[0]):
    for j in range(0,y_true.shape[1]):
        if y_true[i][j] == 2 :
            y_true[i][j] = -1


In [8]:
y_true_c = []
for i in range(0,y_true.shape[0]):
    for j in range(0,y_true.shape[1]):
        if y_true[i][j] ==256 :
            continue
        else:
            y_true_c.append(y_true[i][j])
            
y_true_c = np.array(y_true_c,dtype='int16').reshape(-1,1)

# 读取预测结果

In [45]:
path = r'E:\eaasy\Thesis\水体边界提取文章_apply science\副本\修改资料\Data\Data2019_10_20\ProcessData\ClassificationResult_area2\mlpPredictResult.tif'

#gdal.Register()
dataset = gdal.Open(path)    

geoTransform = dataset.GetGeoTransform()

proj = dataset.GetProjection()

cols = dataset.RasterXSize

rows = dataset.RasterYSize

band = dataset.GetRasterBand(1)


mlp = band.ReadAsArray(0,0,cols,rows)

mlp = np.array(mlp,dtype='int16') 

In [46]:
mlp_c = []
for i in range(0,mlp.shape[0]):
    for j in range(0,mlp.shape[1]):
        
        if mlp[i][j] == 0 or mlp[i][j] == 256 or mlp[i][j] == 32767 :
            continue
        else:
            mlp_c.append(mlp[i][j])
           
mlp_c = np.array(mlp_c,dtype='int16').reshape(-1,1)
mlp_fpr,mlp_tpr,mlp_threshold = roc_curve(mlp_c,y_true_c)
mlp_roc_auc =  auc(mlp_fpr,mlp_tpr)
mlp_roc_auc

0.8499616027544873

In [47]:
labels = [1,-1]

classes = ['water','other']
print(classification_report(y_true_c
                            ,mlp_c
                            ,target_names = classes
                            ,labels = labels
                            ,digits = 3))

              precision    recall  f1-score   support

       water      0.708     0.595     0.647      7781
       other      0.992     0.995     0.993    371164

    accuracy                          0.987    378945
   macro avg      0.850     0.795     0.820    378945
weighted avg      0.986     0.987     0.986    378945



# 随机森林

In [48]:
path = r'E:\eaasy\Thesis\水体边界提取文章_apply science\副本\修改资料\Data\Data2019_10_20\ProcessData\ClassificationResult_area2\RandomForestPredictResult.tif'

#gdal.Register()
dataset = gdal.Open(path)    

geoTransform = dataset.GetGeoTransform()

proj = dataset.GetProjection()

cols = dataset.RasterXSize

rows = dataset.RasterYSize

band = dataset.GetRasterBand(1)


rfc= band.ReadAsArray(0,0,cols,rows)

rfc = np.array(rfc,dtype='int16') 

In [49]:
rfc_c = []
for i in range(0,rfc.shape[0]):
    for j in range(0,rfc.shape[1]):
        
        if rfc[i][j] == 0 or rfc[i][j] == 256  or rfc[i][j] == 32767:
            continue
        else:
         
                rfc_c.append(rfc[i][j])
           
               
rfc_c = np.array(rfc_c,dtype='int16').reshape(-1,1)
rfc_fpr,rfc_tpr,rfc_threshold = roc_curve(rfc_c,y_true_c)
rfc_roc_auc =  auc(rfc_fpr,rfc_tpr)

print(rfc_roc_auc)

0.9731497484459866


In [50]:
labels = [1,-1]

classes = ['water','other']
print(classification_report(y_true_c
                            ,rfc_c
                            ,target_names = classes
                            ,labels = labels
                            ,digits = 3))

              precision    recall  f1-score   support

       water      0.962     0.241     0.385      7781
       other      0.984     1.000     0.992    371164

    accuracy                          0.984    378945
   macro avg      0.973     0.620     0.688    378945
weighted avg      0.984     0.984     0.980    378945



# svm

In [51]:
path = r'E:\eaasy\Thesis\水体边界提取文章_apply science\副本\修改资料\Data\Data2019_10_20\ProcessData\ClassificationResult_area2\svmPredictResult.tif'

#gdal.Register()
dataset = gdal.Open(path)    

geoTransform = dataset.GetGeoTransform()

proj = dataset.GetProjection()

cols = dataset.RasterXSize

rows = dataset.RasterYSize

band = dataset.GetRasterBand(1)


svm = band.ReadAsArray(0,0,cols,rows)

svm = np.array(svm,dtype='int16') 

In [52]:
svm_c = []
for i in range(0,svm.shape[0]):
    for j in range(0,svm.shape[1]):
        
        if svm[i][j] == 0  or svm[i][j] == 256 or  svm[i][j] == 32767 :
            continue
        else:
            svm_c.append(svm[i][j])
svm_c = np.array(svm_c,dtype='int16').reshape(-1,1)

svm_fpr,svm_tpr,svm_threshold = roc_curve(svm_c,y_true_c)
svm_roc_auc =  auc(svm_fpr,svm_tpr)

print(svm_roc_auc)

0.7894852448529223


In [54]:
labels = [1,-1]

classes = ['water','other']
print(classification_report(y_true_c
                            ,svm_c
                            ,target_names = classes
                            ,labels = labels
                            ,digits = 3))

              precision    recall  f1-score   support

       water      0.585     0.712     0.642      7781
       other      0.994     0.989     0.992    371164

    accuracy                          0.984    378945
   macro avg      0.789     0.851     0.817    378945
weighted avg      0.986     0.984     0.984    378945



In [55]:
path = r'E:\eaasy\Thesis\水体边界提取文章_apply science\副本\修改资料\Data\Data2019_10_20\ProcessData\ClassificationResult_area2\xgboostPredictResult.tif'

#gdal.Register()
dataset = gdal.Open(path)    

geoTransform = dataset.GetGeoTransform()

proj = dataset.GetProjection()

cols = dataset.RasterXSize

rows = dataset.RasterYSize

band = dataset.GetRasterBand(1)


xgboost = band.ReadAsArray(0,0,cols,rows)

xgboost = np.array(xgboost,dtype='int16') 

In [56]:
xgboost_c = []
for i in range(0,xgboost.shape[0]):
    for j in range(0,xgboost.shape[1]):
        
        if xgboost[i][j] == 0 or xgboost[i][j] == 256  or  xgboost[i][j] == 32767:
            continue
        else:
            
                xgboost_c.append(xgboost[i][j] )
xgboost_c = np.array(xgboost_c,dtype='int16').reshape(-1,1)

In [57]:
xgboost_fpr,xgboost_tpr,xgboost_threshold = roc_curve(xgboost_c,y_true_c)
xgboost_roc_auc =  auc(xgboost_fpr,xgboost_tpr)

print(xgboost_roc_auc)

0.9724362008753742


In [58]:
labels = [1,-1]

classes = ['water','other']
print(classification_report(y_true_c
                            ,xgboost_c
                            ,target_names = classes
                            ,labels = labels
                            ,digits = 3))

              precision    recall  f1-score   support

       water      0.961     0.221     0.360      7781
       other      0.984     1.000     0.992    371164

    accuracy                          0.984    378945
   macro avg      0.972     0.611     0.676    378945
weighted avg      0.983     0.984     0.979    378945



# 逻辑回归

In [59]:
path = r'E:\eaasy\Thesis\水体边界提取文章_apply science\副本\修改资料\Data\Data2019_10_20\ProcessData\ClassificationResult_area2\LogisticRegressionPredictResult.tif'

#gdal.Register()
dataset = gdal.Open(path)    

geoTransform = dataset.GetGeoTransform()

proj = dataset.GetProjection()

cols = dataset.RasterXSize

rows = dataset.RasterYSize

band = dataset.GetRasterBand(1)


lr = band.ReadAsArray(0,0,cols,rows)

lr = np.array(lr,dtype='int16') 

In [60]:
lr_c = []
for i in range(0,lr.shape[0]):
    for j in range(0,lr.shape[1]):
        
        if lr[i][j] == 0 or lr[i][j] == 256 or lr[i][j] == 32767:
            continue
        else:
           
             
                lr_c.append(lr[i][j])

lr_c = np.array(lr_c,dtype='int16').reshape(-1,1)

In [61]:
lr_fpr,lr_tpr,lr_threshold = roc_curve(lr_c,y_true_c)
lr_roc_auc =  auc(lr_fpr,lr_tpr)

print(lr_roc_auc)

0.9291698131424229


In [63]:
labels = [1,-1]

classes = ['water','other']
print(classification_report(y_true_c
                            ,lr_c
                            ,target_names = classes
                            ,labels = labels
                            ,digits = 3))

              precision    recall  f1-score   support

       water      0.869     0.498     0.633      7781
       other      0.990     0.998     0.994    371164

    accuracy                          0.988    378945
   macro avg      0.929     0.748     0.813    378945
weighted avg      0.987     0.988     0.987    378945



# dt

In [64]:
path = r'E:\eaasy\Thesis\水体边界提取文章_apply science\副本\修改资料\Data\Data2019_10_20\ProcessData\ClassificationResult_area2\DecisionTreePredictResult.tif'

#gdal.Register()
dataset = gdal.Open(path)    

geoTransform = dataset.GetGeoTransform()

proj = dataset.GetProjection()

cols = dataset.RasterXSize

rows = dataset.RasterYSize

band = dataset.GetRasterBand(1)


dt = band.ReadAsArray(0,0,cols,rows)

dt = np.array(dt,dtype='int16') 

In [65]:
dt_c = []
for i in range(0,dt.shape[0]):
    for j in range(0,dt.shape[1]):
        
        if dt[i][j] == 0 :
            continue
        else:
            
                dt_c.append(dt[i][j])
            
                
            
            

dt_c = np.array(dt_c,dtype='int16').reshape(-1,1)


In [66]:
dt_fpr,dt_tpr,dt_threshold = roc_curve(dt_c,y_true_c)
dt_roc_auc =  auc(dt_fpr,dt_tpr)

print(dt_roc_auc)

0.9650037937793787


In [67]:
labels = [1,-1]

classes = ['water','other']
print(classification_report(y_true_c
                            ,dt_c
                            ,target_names = classes
                            ,labels = labels
                            ,digits = 3))

              precision    recall  f1-score   support

       water      0.948     0.144     0.250      7781
       other      0.982     1.000     0.991    371164

    accuracy                          0.982    378945
   macro avg      0.965     0.572     0.621    378945
weighted avg      0.982     0.982     0.976    378945



# adaptive

In [68]:
path = r'E:\eaasy\Thesis\水体边界提取文章_apply science\副本\修改资料\Data\Data2019_10_20\ProcessData\ClassificationResult_area2\AdaptiveThresholdResult.tif'

#gdal.Register()
dataset = gdal.Open(path)    

geoTransform = dataset.GetGeoTransform()

proj = dataset.GetProjection()

cols = dataset.RasterXSize

rows = dataset.RasterYSize

band = dataset.GetRasterBand(1)


adaptive = band.ReadAsArray(0,0,cols,rows)

adaptive = np.array(adaptive,dtype='int16') 

In [69]:
adaptive_c = []
for i in range(0,adaptive.shape[0]):
    for j in range(0,adaptive.shape[1]):
        
        if adaptive[i][j] == 0 or adaptive[i][j] == 32767:
            continue
        else:
            adaptive_c.append(adaptive[i][j])
            

adaptive_c = np.array(adaptive_c,dtype='int16').reshape(-1,1)


In [70]:
adaptive_fpr,adaptive_tpr,adaptive_threshold = roc_curve(adaptive_c,y_true_c)
adaptive_roc_auc =  auc(adaptive_fpr,adaptive_tpr)

print(adaptive_roc_auc)

0.5066423934134183


In [71]:
labels = [1,-1]

classes = ['water','other']
print(classification_report(y_true_c
                            ,adaptive_c
                            ,target_names = classes
                            ,labels = labels
                            ,digits = 3))

              precision    recall  f1-score   support

       water      0.027     0.670     0.052      7781
       other      0.986     0.496     0.660    371164

    accuracy                          0.499    378945
   macro avg      0.507     0.583     0.356    378945
weighted avg      0.967     0.499     0.647    378945



# COUSTOM


In [72]:
path = r'E:\eaasy\Thesis\水体边界提取文章_apply science\副本\修改资料\Data\Data2019_10_20\ProcessData\ClassificationResult_area2\CustomThresholdResult.tif'

#gdal.Register()
dataset = gdal.Open(path)    

geoTransform = dataset.GetGeoTransform()

proj = dataset.GetProjection()

cols = dataset.RasterXSize

rows = dataset.RasterYSize

band = dataset.GetRasterBand(1)


custom = band.ReadAsArray(0,0,cols,rows)

custom = np.array(custom,dtype='int16') 

In [73]:
custom_c = []
for i in range(0,custom.shape[0]):
    for j in range(0,custom.shape[1]):
        
        if custom[i][j] == 0 or custom[i][j] == 32767:
            continue
        else:
            custom_c.append(custom[i][j])
            

custom_c = np.array(custom_c,dtype='int16').reshape(-1,1)


In [74]:
custom_fpr,custom_tpr,custom_threshold = roc_curve(custom_c,y_true_c)
custom_roc_auc =  auc(custom_fpr,custom_tpr)

print(custom_roc_auc)

0.6259754897665389


In [76]:
labels = [1,-1]

classes = ['water','other']
print(classification_report(y_true_c
                            ,custom_c
                            ,target_names = classes
                            ,labels = labels
                            ,digits = 3))

              precision    recall  f1-score   support

       water      0.256     0.825     0.391      7781
       other      0.996     0.950     0.972    371164

    accuracy                          0.947    378945
   macro avg      0.626     0.887     0.681    378945
weighted avg      0.981     0.947     0.960    378945



# oust

In [77]:
path = r'E:\eaasy\Thesis\水体边界提取文章_apply science\副本\修改资料\Data\Data2019_10_20\ProcessData\ClassificationResult_area2\OustThresholdResult.tif'

#gdal.Register()
dataset = gdal.Open(path)    

geoTransform = dataset.GetGeoTransform()

proj = dataset.GetProjection()

cols = dataset.RasterXSize

rows = dataset.RasterYSize

band = dataset.GetRasterBand(1)


oust = band.ReadAsArray(0,0,cols,rows)

oust = np.array(oust,dtype='int16') 

In [78]:
oust_c = []
for i in range(0,oust.shape[0]):
    for j in range(0,oust.shape[1]):
        
        if oust[i][j] == 0 or oust[i][j] == 32767:
            continue
        else:
            oust_c.append(oust[i][j])
            

oust_c = np.array(oust_c,dtype='int16').reshape(-1,1)


In [79]:
oust_fpr,oust_tpr,oust_threshold = roc_curve(oust_c,y_true_c)
oust_roc_auc =  auc(oust_fpr,oust_tpr)

print(oust_roc_auc)

0.9215371882299671


In [80]:
labels = [1,-1]

classes = ['water','other']
print(classification_report(y_true_c
                            ,oust_c
                            ,target_names = classes
                            ,labels = labels
                            ,digits = 3))

              precision    recall  f1-score   support

       water      0.856     0.361     0.508      7781
       other      0.987     0.999     0.993    371164

    accuracy                          0.986    378945
   macro avg      0.922     0.680     0.750    378945
weighted avg      0.984     0.986     0.983    378945

